# Contrastive Fine-Tuning with Hard Negatives & Embedding Model Selection

## Overview
This notebook walks through fine-tuning sentence embeddings and selecting production models:
1. **Contrastive Fine-Tuning**: MultipleNegativesRankingLoss (MNRL) and in-batch negative mechanics.
2. **Hard Negative Mining**: Using BM25 lexical distractors to construct high-quality training triplets.
3. **Model Training**: Fine-tuning with `SentenceTransformerTrainer`.
4. **Evaluation**: Measuring Recall@k and nDCG@10 before and after fine-tuning.
5. **Model Selection**: Comparing open-source vs. proprietary commercial embedding models.
6. **Production Checklist**: Best practices for chunking, prefixes, dimensions, and indexing.
7. **Key Takeaways**: Summary of sentence representation techniques.

In [1]:
# Cell 1: Environment Setup & Pinned Library Versions
import os
import sys
import time

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import torch
import transformers
import sentence_transformers
import rank_bm25
import faiss
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset

print(f"Python Version:               {sys.version.split()[0]}")
print(f"PyTorch Version:              {torch.__version__}")
print(f"Transformers Version:         {transformers.__version__}")
print(f"Sentence-Transformers Version: {sentence_transformers.__version__}")
print(f"Rank-BM25 Version:            {getattr(rank_bm25, '__version__', '0.2.2')}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary Acceleration Device:  {device}")


/Users/apujari/Documents/courses/AgenticAI/code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python Version:               3.12.15
PyTorch Version:              2.14.1
Transformers Version:         5.19.0
Sentence-Transformers Version: 6.1.0
Rank-BM25 Version:            0.2.2
Primary Acceleration Device:  mps


---
## 1. Contrastive Fine-Tuning & MNRL

### 1.1 Why Fine-Tune Embedding Models?
Off-the-shelf embedding models perform well on general open-domain text, but often fail on specialized domains (e.g. legal contracts, medical reports, or technical research papers). Fine-tuning adapts the vector space so that:
- Domain-specific terms (e.g., `Scaled Dot-Product Attention`, `RoPE`, `GeGLU`) align closely with their explanatory definitions.
- Confusing distractors (e.g. French BLEU score vs. German BLEU score) are pushed far apart.

### 1.2 MultipleNegativesRankingLoss (MNRL)
Given a batch of $B$ triplets $(a_i, p_i, n_i)$ where $a_i$ is the anchor query, $p_i$ is the positive document, and $n_i$ is a mined hard negative:
$$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{e^{\text{sim}(a_i, p_i)/\tau}}{e^{\text{sim}(a_i, p_i)/\tau} + e^{\text{sim}(a_i, n_i)/\tau} + \sum_{j \neq i} \left( e^{\text{sim}(a_i, p_j)/\tau} + e^{\text{sim}(a_i, n_j)/\tau} \right)}$$
For each anchor $a_i$, **every other document in the batch** ($p_j$ and $n_j$ for $j \neq i$) acts as a negative candidate.

---
## 2. Hard Negative Mining with BM25

A common mistake in fine-tuning is using random negatives. Random negatives (e.g. pairing a question about Transformers with a paragraph about pizza) are too easy — the model learns nothing.

**Hard negatives** are passages that:
1. Share high vocabulary overlap with the query (they look relevant to lexical search).
2. Do **not** actually answer the query.

We use **BM25** to mine the top lexical distractor for each query:

In [2]:
# Cell 2: Load Corpus & Mine BM25 Hard Negatives
import json
from rank_bm25 import BM25Okapi

possible_paths = [
    os.path.abspath("../../data/papers_chunks.jsonl"),
    os.path.abspath("../data/papers_chunks.jsonl"),
    os.path.abspath("./data/papers_chunks.jsonl"),
    os.path.expanduser("~/Documents/courses/AgenticAI/data/papers_chunks.jsonl")
]

corpus_path = None
for p in possible_paths:
    if os.path.exists(p):
        corpus_path = p
        break

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus_chunks = [json.loads(line) for line in f]

corpus_texts = [c["text"] for c in corpus_chunks]
print(f"Loaded {len(corpus_texts)} corpus passages.")

def tokenize_text(t):
    return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenize_text(doc) for doc in corpus_texts])

# 12 Verified Query-Positive passage pairs
qa_pairs = [
    ("What BLEU score did Transformer achieve on WMT 2014 English-to-German?", 2),
    ("How is scaled dot product attention computed with queries, keys, and values?", 8),
    ("Why does multi-head attention project queries, keys, and values into multiple subspaces?", 9),
    ("What sinusoidal functions are used for positional encoding in the Transformer?", 14),
    ("What is the computational complexity per layer of self-attention compared to recurrent layers?", 15),
    ("How is the Masked Language Model MLM pre-training task designed in BERT?", 35),
    ("What is Next Sentence Prediction NSP and why was it included in BERT pre-training?", 48),
    ("How is fine-tuning BERT performed on downstream classification and question answering tasks?", 39),
    ("What are the key differences between feature-based and fine-tuning approaches with BERT?", 45),
    ("How does in-context few-shot learning work without updating model weights?", 60),
    ("What Common Crawl filtering and deduplication was applied to create the training dataset?", 63),
    ("What batch size and learning rate scaling schedule was used for training large language models?", 64)
]

# Mine BM25 Hard Negatives (top BM25 match that is NOT the positive chunk)
mined_triplets = []
for q, pos_idx in qa_pairs:
    bm25_scores = bm25.get_scores(tokenize_text(q))
    ranked_indices = np.argsort(-bm25_scores)
    # Pick the top distractor that is NOT the positive passage
    neg_idx = [idx for idx in ranked_indices if idx != pos_idx][0]
    
    mined_triplets.append({
        "anchor": q,
        "positive": corpus_texts[pos_idx],
        "negative": corpus_texts[neg_idx],
        "pos_id": pos_idx,
        "neg_id": neg_idx
    })

print(f"Mined {len(mined_triplets)} high-quality triplets with BM25 lexical distractors.")
print(f"\nSample Triplet for Query 0 ('{mined_triplets[0]['anchor']}'):")
print(f"  Positive (Chunk {mined_triplets[0]['pos_id']}): {mined_triplets[0]['positive'][:110]}...")
print(f"  Negative (Chunk {mined_triplets[0]['neg_id']}): {mined_triplets[0]['negative'][:110]}...")


Loaded 125 corpus passages.
Mined 12 high-quality triplets with BM25 lexical distractors.

Sample Triplet for Query 0 ('What BLEU score did Transformer achieve on WMT 2014 English-to-German?'):
  Positive (Chunk 2): ## **Abstract** 

The dominant sequence transduction models are based on complex recurrent or convolutional ne...
  Negative (Chunk 21): ## **6.1 Machine Translation** 

On the WMT 2014 English-to-German translation task, the big transformer model...


---
## 3. Fine-Tuning with Sentence Transformers

We split our triplets into **Train** (8 queries) and **Test** (4 held-out queries).
Then, we fine-tune `sentence-transformers/all-MiniLM-L6-v2` for 3 epochs using `MultipleNegativesRankingLoss`.

In [3]:
# Cell 3: Pre-Training Evaluation Baseline on Held-Out Test Queries
from sentence_transformers import SentenceTransformer

train_triplets = mined_triplets[:8]
test_triplets = mined_triplets[8:]
print(f"Train split: {len(train_triplets)} triplets | Test split: {len(test_triplets)} triplets")

base_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def evaluate_test_queries(model):
    """Evaluate test queries against the full 125-chunk corpus."""
    doc_embs = model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=False)
    r1_list, r5_list, ndcg_list = [], [], []
    
    for item in test_triplets:
        q_text = item["anchor"]
        pos_id = item["pos_id"]
        q_emb = model.encode(q_text, normalize_embeddings=True, show_progress_bar=False)
        sims = np.dot(doc_embs, q_emb)
        ranked = list(np.argsort(-sims))
        
        r1_list.append(1.0 if ranked[0] == pos_id else 0.0)
        r5_list.append(1.0 if pos_id in ranked[:5] else 0.0)
        
        if pos_id in ranked[:10]:
            rank_pos = ranked[:10].index(pos_id)
            ndcg_list.append(1.0 / np.log2(rank_pos + 2))
        else:
            ndcg_list.append(0.0)
            
    return {
        "Recall@1": float(np.mean(r1_list)),
        "Recall@5": float(np.mean(r5_list)),
        "nDCG@10": float(np.mean(ndcg_list))
    }

pre_metrics = evaluate_test_queries(base_model)
print("\n--- Pre-Fine-Tuning Baseline Metrics (Held-Out Test Queries) ---")
for k, v in pre_metrics.items():
    print(f"  {k:<12}: {v:.4f}")


Train split: 8 triplets | Test split: 4 triplets


Loading weights:   0%|                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 15526.08it/s]


--- Pre-Fine-Tuning Baseline Metrics (Held-Out Test Queries) ---
  Recall@1    : 0.2500
  Recall@5    : 0.7500
  nDCG@10     : 0.5655


In [4]:
# Cell 4: Train with SentenceTransformerTrainer and MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

# Prepare HuggingFace Dataset
train_dataset = Dataset.from_dict({
    "anchor":   [t["anchor"]   for t in train_triplets],
    "positive": [t["positive"] for t in train_triplets],
    "negative": [t["negative"] for t in train_triplets]
})

output_dir = "scratch/finetuned_all_minilm"
os.makedirs(output_dir, exist_ok=True)

loss_fn = MultipleNegativesRankingLoss(base_model)

training_args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss_fn
)

t0_train = time.time()
trainer.train()
train_duration = time.time() - t0_train
print(f"\nFine-tuning completed in {train_duration:.2f} seconds!")


The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Computing widget examples:   0%|                    | 0/1 [00:00<?, ?example/s]

/Users/apujari/Documents/courses/AgenticAI/code/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.459637
2,0.265687
3,0.846583
4,0.778040
5,0.244225
6,0.861063



Fine-tuning completed in 0.88 seconds!


---
## 4. Before vs. After Evaluation

We evaluate the fine-tuned model on the **held-out test queries** to verify generalization and quantify retrieval gains:

In [5]:
# Cell 5: Post-Fine-Tuning Evaluation and Comparison Table
post_metrics = evaluate_test_queries(base_model)

ft_comparison_df = pd.DataFrame([
    {
        "Model State": "Pre-Fine-Tuning (Base all-MiniLM)",
        "Recall@1": round(pre_metrics["Recall@1"], 3),
        "Recall@5": round(pre_metrics["Recall@5"], 3),
        "nDCG@10":  round(pre_metrics["nDCG@10"], 3)
    },
    {
        "Model State": "Post-Fine-Tuning (Contrastive MNRL + Hard Negatives)",
        "Recall@1": round(post_metrics["Recall@1"], 3),
        "Recall@5": round(post_metrics["Recall@5"], 3),
        "nDCG@10":  round(post_metrics["nDCG@10"], 3)
    }
])

print("="*80)
print(" Before vs. After Fine-Tuning Performance on Held-Out Test Set")
print("="*80)
print(ft_comparison_df.to_string(index=False))


 Before vs. After Fine-Tuning Performance on Held-Out Test Set
                                         Model State  Recall@1  Recall@5  nDCG@10
                   Pre-Fine-Tuning (Base all-MiniLM)      0.25      0.75    0.565
Post-Fine-Tuning (Contrastive MNRL + Hard Negatives)      0.25      0.75    0.565


---
## 5. Choosing an Embedding Model

The table below summarizes the landscape of open and proprietary embedding models:

| Model | Backbone | Access | Typical Dims | Max Context | Strengths & Optimal Use Case |
|---|---|---|---|---|---|
| **S-BERT (all-MiniLM)** | Small BERT-class encoder (22M params) | Open (Apache 2.0) | 384d | 512 tokens | **Tiny, ultra-fast baseline**; ideal for mobile, CPU-only, or high-throughput microservices. |
| **E5 / BGE / GTE** | BERT-class encoder (0.1B–0.6B params) | Open (MIT / Apache) | 384d / 768d / 1024d | 512 tokens | **Strong default workhorse**; cheap to self-host, top accuracy on MTEB retrieval. |
| **BGE-M3** | XLM-RoBERTa encoder (560M params) | Open (MIT) | 1024d | **8,192 tokens** | **Dense + Sparse + Multi-vector** in one model; multilingual across 100+ languages. |
| **ModernBERT-embed** | ModernBERT encoder (150M–395M params) | Open (Apache 2.0) | 768d | **8,192 tokens** | Modern architecture with RoPE, FlashAttention, and 8k context at BERT-like latency. |
| **Qwen3-Embedding** | Decoder LLM (0.6B / 4B / 8B params) | Open (Apache 2.0) | 1024d–4096d | **32,768 tokens** | **Top MMTEB leaderboard quality**; instruction-conditioned, native Matryoshka support. |
| **OpenAI text-embedding-3** | Proprietary LLM backbone | API | 512d–3072d (MRL) | 8,191 tokens | **Matryoshka dimensions parameter**; convenient API, strong English retrieval. |
| **Gemini Embedding 2** | Proprietary multimodal LLM | API | 768d–3072d | 8,192 tokens | **Native multimodal retrieval** (text, images, audio, video, PDF) in a shared geometric space. |
| **Cohere Embed v4 / Voyage** | Proprietary LLM backbones | API | 1024d | 8,000 tokens | Specialized enterprise search, compression, and high-precision financial/legal retrieval. |

---
## 6. Production Checklist

Deploying embedding models in production requires following these **7 essential rules**:

1. **Follow the Model Card (Prefixes)**:
   - Always verify if your chosen model requires asymmetric prefixes (`query: ` vs. `passage: ` in E5, task prompts in BGE).
   - *A wrong or missing prefix silently degrades Recall by 15–30%!*
2. **Normalize Vectors**:
   - $L_2$-normalize all vectors so that inner product (`IndexFlatIP` in FAISS) exactly equals cosine similarity.
3. **Match Chunk Size to Context Window**:
   - Do not stuff 2,000 tokens into a 512-token BERT model; sentences beyond token 512 are silently truncated.
   - Long chunks dilute vector representation; 250–500 tokens is typically optimal for retrieval.
4. **Choose Dimensions Deliberately (Matryoshka & Quantization)**:
   - Use Matryoshka truncation (e.g. 128d or 256d) to cut index RAM by $3\times\text{--}6\times$.
   - Apply int8 quantization for high-throughput vector databases.
5. **Never Mix Models in One Index**:
   - Vector spaces trained with different models have incompatible geometric coordinates.
   - *Changing your embedding model requires re-embedding your entire corpus.*
6. **Implement Hybrid Search + Cross-Encoder Reranking**:
   - Combine dense retrieval with BM25 using Reciprocal Rank Fusion (RRF).
   - Pass the top 50 candidates to a Cross-Encoder for precision reranking before LLM generation.
7. **Build Domain-Specific Evaluation Sets**:
   - Do not rely solely on public MTEB scores. Build a golden set of 20–50 real user queries and measure Recall@5 and nDCG@10 on your own data.

---
## 7. Key Takeaways

- **Evolution**: Sentence representations advanced from static bag-of-words and Word2Vec to S-BERT's bi-encoders, and now to **multi-billion parameter decoder LLM embedders**.
- **The Modern Recipe**: Multi-stage contrastive pre-training (InfoNCE) on massive weakly supervised text pairs, followed by supervised fine-tuning with **mined hard negatives**.
- **Flexibility**: Matryoshka Representation Learning (MRL) allows single models to serve multiple latency, memory, and cost tiers without retraining.
- **Hybrid Synergy**: Dense semantic search and lexical exact-match search are complementary, not competing; modern production systems fuse both with **RRF**.
- **Validation**: Public benchmarks guide initial model shortlists, but private domain evaluation decides what ships to production.

---
## 8. Summary & Exercises

### Question 1
Why are in-batch negatives called "free negatives" in MultipleNegativesRankingLoss (MNRL)?
<details>
<summary><b>View Answer</b></summary>
In a mini-batch of $B$ pairs, computing the $(B \times B)$ cross-similarity matrix produces 1 positive pair and $B-1$ negative pairs for every anchor query without requiring extra forward passes or database lookups.
</details>

### Question 2
What happens if you query a vector database containing embeddings from `all-MiniLM-L6-v2` using a query encoded with `bge-small-en-v1.5`?
<details>
<summary><b>View Answer</b></summary>
Retrieval fails completely. Even if both models output 384-dimensional vectors, their internal geometric spaces are oriented completely differently. Similarity comparisons between different models are mathematically meaningless.
</details>

### Question 3
Why does combining dense retrieval with BM25 using Reciprocal Rank Fusion (RRF) consistently outperform either method alone in real-world applications?
<details>
<summary><b>View Answer</b></summary>
Dense retrieval excels at semantic paraphrasing and conceptual abstraction, but struggles with exact IDs, model numbers, and acronyms. BM25 excels at exact keyword matches but fails on vocabulary mismatch. RRF merges both rank lists robustly, ensuring that exact term queries and conceptual queries both return top relevant hits.
</details>